# Conceptual Intuition: Nearest Neighbors & Disjoint Clusters
No Chroma server needed for this one — pure embedding-space math on the same dataset, so you can confirm the *expected* behavior numerically before (or instead of) eyeballing it in a scatter plot. Run this BEFORE or alongside the ingestion notebook.

In [ ]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath("../wrapper_fix"))
sys.path.append(os.path.abspath("../inhouse_rag_capstone"))
from inhouse_wrappers import InHouseEmbeddings

embedder = InHouseEmbeddings()  # calls the real get_embeddings() under the hood --
                                  # see manual_curl_examples.sh if you want the raw HTTP shape

df = pd.read_excel("../dataset/practice_dataset.xlsx")
vectors = np.array(embedder.embed_documents(df["text"].tolist()))
print(df.shape, vectors.shape)

## 1. Cosine similarity matrix

In [ ]:
def cosine_sim_matrix(vecs):
    norm = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    return norm @ norm.T

sim = cosine_sim_matrix(vectors)
ids = df["id"].tolist()

## 2. Check near-duplicates: do they show the HIGHEST similarity to their parent?

In [ ]:
dup_pairs = [("dup_01", "coo_01"), ("dup_02", "fin_01"), ("dup_03", "pro_01")]
for dup_id, parent_id in dup_pairs:
    i, j = ids.index(dup_id), ids.index(parent_id)
    same_cluster_topic = df.loc[df["id"] == parent_id, "topic"].iloc[0]
    other_same_topic_idx = [k for k, t in enumerate(df["topic"]) if t == same_cluster_topic and ids[k] != dup_id]
    sims_to_topic = sim[i, other_same_topic_idx]
    print(f"{dup_id} vs its paraphrase target {parent_id}: similarity = {sim[i,j]:.4f}")
    print(f"  vs rest of '{same_cluster_topic}' cluster: mean={sims_to_topic.mean():.4f}, max={sims_to_topic.max():.4f}")
    print(f"  -> paraphrase similarity {'IS' if sim[i,j] >= sims_to_topic.max() else 'is NOT'} the highest in-cluster similarity\n")

## 3. Check bridge documents: do they show split similarity to BOTH parent clusters?

In [ ]:
bridge_checks = [("brg_01", "sports", "medicine"), ("brg_02", "travel", "history")]
for bridge_id, topic_a, topic_b in bridge_checks:
    i = ids.index(bridge_id)
    idx_a = [k for k, t in enumerate(df["topic"]) if t == topic_a]
    idx_b = [k for k, t in enumerate(df["topic"]) if t == topic_b]
    idx_other = [k for k, t in enumerate(df["topic"]) if t not in (topic_a, topic_b) and df["type"].iloc[k] == "core"]
    print(f"{bridge_id} mean similarity to '{topic_a}': {sim[i, idx_a].mean():.4f}")
    print(f"{bridge_id} mean similarity to '{topic_b}': {sim[i, idx_b].mean():.4f}")
    print(f"{bridge_id} mean similarity to all OTHER core clusters: {sim[i, idx_other].mean():.4f}")
    print(f"  -> expect both parent-cluster numbers clearly higher than the 'other clusters' number\n")

## 4. Check outliers: do they show LOW similarity to everything?

In [ ]:
outlier_ids = ["out_01", "out_02"]
core_idx = [k for k, t in enumerate(df["type"]) if t == "core"]
for out_id in outlier_ids:
    i = ids.index(out_id)
    print(f"{out_id} mean similarity to all core docs: {sim[i, core_idx].mean():.4f}")
    print(f"{out_id} max similarity to any single core doc: {sim[i, core_idx].max():.4f}")
print("\nFor comparison, mean WITHIN-cluster similarity (cooking):")
coo_idx = [k for k, t in enumerate(df["topic"]) if t == "cooking"]
coo_sim = sim[np.ix_(coo_idx, coo_idx)]
print(np.mean(coo_sim[np.triu_indices_from(coo_sim, k=1)]))
print("\n-> outlier similarity numbers should be noticeably lower than the within-cluster number above")

## 5. PCA visualization, colored by topic AND by type

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(vectors)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for topic in df["topic"].unique():
    idx = df.index[df["topic"] == topic]
    axes[0].scatter(coords[idx, 0], coords[idx, 1], label=topic, s=40)
axes[0].set_title("Colored by topic")
axes[0].legend(fontsize=7, loc="best")

type_colors = {"core": "lightgray", "near_duplicate": "red", "bridge": "blue", "outlier": "black"}
for t, c in type_colors.items():
    idx = df.index[df["type"] == t]
    axes[1].scatter(coords[idx, 0], coords[idx, 1], label=t, c=c, s=60 if t != "core" else 30)
axes[1].set_title("Colored by type (watch where red/blue/black land)")
axes[1].legend(fontsize=9)
plt.show()

## 6. A simple recall@k check: does querying within a cluster return same-cluster neighbors?

In [ ]:
def recall_at_k(k=5):
    results = {}
    for topic in clusters_to_check:
        idx = [i for i, t in enumerate(df["topic"]) if t == topic]
        hits, total = 0, 0
        for i in idx:
            sims = sim[i].copy()
            sims[i] = -1  # exclude self
            top_k = np.argsort(sims)[-k:]
            hits += sum(1 for j in top_k if df["topic"].iloc[j] == topic)
            total += k
        results[topic] = hits / total
    return results

clusters_to_check = [t for t in df["topic"].unique() if df[df["topic"] == t].shape[0] >= 6]
recall_scores = recall_at_k(k=3)
for topic, score in recall_scores.items():
    print(f"{topic:12s} recall@3 = {score:.2f}")
print("\nA score near 1.0 means: querying any document in that cluster reliably "
      "surfaces other members of the SAME cluster as nearest neighbors -- exactly "
      "what good retrieval should do. Low scores flag a cluster that's not well "
      "separated, worth a closer look in Chroma Navigator's Visualize tab.")

## Teaser exercise
Run section 6 again but compute recall@k including the bridge and outlier documents in the candidate pool (not excluded). Does the sports cluster's recall@3 drop because brg_01 (sports+medicine) keeps showing up as a 'neighbor' that isn't a clean topic match? That's the exact kind of ambiguity real-world RAG corpora are full of — this dataset just lets you see it happen on demand.